Câu 1: Đọc và lọc dữ liệu nước Đức

In [1]:
# Thông tin sinh viên
print("Lê Quốc Trung - 6451071082")

import pandas as pd
import warnings
import io
import requests

warnings.filterwarnings("ignore")

# Đọc dữ liệu
try:
    df = pd.read_excel("Online Retail.xlsx")
except Exception:
    file_id = "12q4inbGbTS1UKs9sHA1X87vSRV8feEHV"
    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    response = requests.get(download_url)
    df = pd.read_excel(io.BytesIO(response.content))

# Tiền xử lý cơ bản
df["Description"] = df["Description"].astype(str).str.strip()
df = df.dropna(subset=["InvoiceNo"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.contains('C')]
if 'POSTAGE' in df["Description"].values:
    df = df[df["Description"] != 'POSTAGE']

# Lọc riêng dữ liệu nước Đức ('Germany')
basket_germany = (df[df["Country"] == 'Germany']
                  .groupby(["InvoiceNo", "Description"])["Quantity"]
                  .sum().unstack().reset_index().fillna(0).set_index('InvoiceNo'))

basket_sets_germany = basket_germany.map(lambda qty: qty >= 1).astype(bool)

print("--- CÂU 1 (ĐỨC): LỌC DỮ LIỆU THÀNH CÔNG ---")
print(f"Kích thước giỏ hàng nước Đức: {basket_sets_germany.shape}")
display(basket_sets_germany.head())

Họ và tên: Trần Minh Tấn
MSSV: 6451071071
--- CÂU 1 (ĐỨC): LỌC DỮ LIỆU THÀNH CÔNG ---
Kích thước giỏ hàng nước Đức: (449, 1694)


Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE SKULLS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536840,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536861,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536967,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
536983,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Câu 2: Tìm tập phổ biến Apriori với min_sup = 5%

In [2]:
# Thông tin sinh viên
print("Lê Quốc Trung - 6451071082")

import pandas as pd
import warnings
from mlxtend.frequent_patterns import apriori
import io
import requests

warnings.filterwarnings("ignore")

try:
    df = pd.read_excel("Online Retail.xlsx")
except Exception:
    file_id = "12q4inbGbTS1UKs9sHA1X87vSRV8feEHV"
    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    response = requests.get(download_url)
    df = pd.read_excel(io.BytesIO(response.content))

df["Description"] = df["Description"].astype(str).str.strip()
df = df.dropna(subset=["InvoiceNo"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.contains('C')]
if 'POSTAGE' in df["Description"].values:
    df = df[df["Description"] != 'POSTAGE']

basket_germany = (df[df["Country"] == 'Germany']
                  .groupby(["InvoiceNo", "Description"])["Quantity"]
                  .sum().unstack().reset_index().fillna(0).set_index('InvoiceNo'))

basket_sets_germany = basket_germany.map(lambda qty: qty >= 1).astype(bool)

# Tìm tập phổ biến bằng Apriori với min_sup = 5% (0.05)
frequent_itemsets_germany = apriori(basket_sets_germany, min_support=0.05, use_colnames=True)

print("--- CÂU 2 (ĐỨC): TẬP PHỔ BIẾN APRIORI (min_sup = 5%) ---")
print(f"Tổng số tập phổ biến: {len(frequent_itemsets_germany)}")
display(frequent_itemsets_germany.head(10))

Họ và tên: Trần Minh Tấn
MSSV: 6451071071
--- CÂU 2 (ĐỨC): TẬP PHỔ BIẾN APRIORI (min_sup = 5%) ---
Tổng số tập phổ biến: 59


,support,itemsets
0,0.104677,(6 RIBBONS RUSTIC CHARM)
1,0.071269,(ALARM CLOCK BAKELIKE PINK)
2,0.066815,(CHARLOTTE BAG APPLES DESIGN)
3,0.051225,(CHILDRENS CUTLERY DOLLY GIRL)
4,0.062361,(COFFEE MUG APPLES DESIGN)
5,0.064588,(FAWN BLUE HOT WATER BOTTLE)
6,0.073497,(GUMBALL COAT RACK)
7,0.057906,(IVORY KITCHEN SCALES)
8,0.064588,(JAM JAR WITH PINK LID)
9,0.093541,(JAM MAKING SET PRINTED)


Câu 3: Tạo luật kết hợp với min_conf = 50%

In [3]:
# Thông tin sinh viên
print("Lê Quốc Trung - 6451071082")

import pandas as pd
import warnings
from mlxtend.frequent_patterns import apriori, association_rules
import io
import requests

warnings.filterwarnings("ignore")

try:
    df = pd.read_excel("Online Retail.xlsx")
except Exception:
    file_id = "12q4inbGbTS1UKs9sHA1X87vSRV8feEHV"
    download_url = f"https://drive.google.com/uc?export=download&id={file_id}"
    response = requests.get(download_url)
    df = pd.read_excel(io.BytesIO(response.content))

df["Description"] = df["Description"].astype(str).str.strip()
df = df.dropna(subset=["InvoiceNo"])
df["InvoiceNo"] = df["InvoiceNo"].astype(str)
df = df[~df["InvoiceNo"].str.contains('C')]
if 'POSTAGE' in df["Description"].values:
    df = df[df["Description"] != 'POSTAGE']

basket_germany = (df[df["Country"] == 'Germany']
                  .groupby(["InvoiceNo", "Description"])["Quantity"]
                  .sum().unstack().reset_index().fillna(0).set_index('InvoiceNo'))

basket_sets_germany = basket_germany.map(lambda qty: qty >= 1).astype(bool)

frequent_itemsets_germany = apriori(basket_sets_germany, min_support=0.05, use_colnames=True)

# Tạo luật kết hợp với min_conf = 50% (0.5)
rules_germany = association_rules(frequent_itemsets_germany, metric="confidence", min_threshold=0.5)

print("--- CÂU 3 (ĐỨC): LUẬT KẾT HỢP (min_conf = 50%) ---")
print(f"Tổng số luật kết hợp: {len(rules_germany)}")
display(rules_germany[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Họ và tên: Trần Minh Tấn
MSSV: 6451071071
--- CÂU 3 (ĐỨC): LUẬT KẾT HỢP (min_conf = 50%) ---
Tổng số luật kết hợp: 8


,antecedents,consequents,support,confidence,lift
0,(PLASTERS IN TIN CIRCUS PARADE),(PLASTERS IN TIN WOODLAND ANIMALS),0.069042,0.584906,4.168613
1,(PLASTERS IN TIN SPACEBOY),(PLASTERS IN TIN WOODLAND ANIMALS),0.062361,0.571429,4.072562
2,(PLASTERS IN TIN WOODLAND ANIMALS),(ROUND SNACK BOXES SET OF4 WOODLAND),0.075724,0.539683,2.163549
3,(RED RETROSPOT CHARLOTTE BAG),(WOODLAND CHARLOTTE BAG),0.060134,0.843750,6.531789
4,(ROUND SNACK BOXES SET OF4 WOODLAND),(ROUND SNACK BOXES SET OF 4 FRUITS),0.133630,0.535714,3.340774
5,(ROUND SNACK BOXES SET OF 4 FRUITS),(ROUND SNACK BOXES SET OF4 WOODLAND),0.133630,0.833333,3.340774
6,(SPACEBOY LUNCH BOX),(ROUND SNACK BOXES SET OF4 WOODLAND),0.071269,0.680851,2.729483
7,(WOODLAND CHARLOTTE BAG),(ROUND SNACK BOXES SET OF4 WOODLAND),0.064588,0.500000,2.004464
